In [2]:
import nltk
import pandas as pd
import random
import difflib
from wordfreq import zipf_frequency
from nltk.corpus import wordnet as wn
random.seed(42)

In [ ]:
#scarico wordnet, lo faccio una volta e basta
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...


True

In [13]:
#prova estrazione da word.net di parole tramite pos punto 1, filtriamo anche per quanto è comune il significato

def extract_words(pos):
    #creo un set per evitare duplicati
    #prelevo parole a seconda del pos (aggettivi,nomi ecc..) utilizzato
    #metto tutto in minuscolo
    #applico i filtri: parole con solo lettere all'interno, lunghe più di 3 lettere, con un significato abbastanza comune
    #se corrispondono lo aggiungo nel set

    words = set()

    for syn in wn.all_synsets(pos=pos):

        for lemma in syn.lemmas():

            word = lemma.name().lower()

            if (
                word.isalpha()
                and len(word) >= 3
                and zipf_frequency(word, "en") >= 2.5
            ):
                words.add(word)

    return list(words)

In [14]:
#prova punto 2 quanti aggettivi estraiamo?
adj_words = sorted(extract_words(wn.ADJ))
print(len(adj_words))

7812


In [36]:
#prova punto 4: funzione per sinonimi
def get_synonym_pair(word, pos):
    # Ricavo sinonimi da WordNet utilizzando solo il synset principale
    # per ridurre la polisemia.
    # Mantengo solo lemmi single-token e diversi dalla parola originale.
    # Per evitare over-representation dei synset più densi
    # seleziono un solo sinonimo casuale ma riproducibile.
    #filtri: sinonimo contente solo lettere e non altri caratteri, lunghezza sinonimo > di 3 lettere,  sinonimo abbastanza comune

    synsets = wn.synsets(word, pos=pos)

    if not synsets:
        return None

    main_synset = synsets[0]

    synonyms = set()

    for lemma in main_synset.lemmas():

        name = lemma.name().lower()

        if (
            name.isalpha()
            and name != word
            and len(name) >= 3
            and zipf_frequency(name, "en") >= 2.5
            and len(wn.synsets(name, pos=pos)) <= 5
        ):

            synonym_synsets = wn.synsets(name, pos=pos)

            # controllo che il senso condiviso sia dominante
            if synonym_synsets[0] == main_synset:
                synonyms.add(name)


    if synonyms:
        synonym = random.choice(list(synonyms))
        return (word, synonym)

    return None

In [37]:
#prova punto 5 creo lista con sinonimi per le 500 parole
synonym_pairs_test = []

for word in adj_words:

    pair = get_synonym_pair(word, wn.ADJ)

    if pair:
        synonym_pairs_test.append(pair)

In [38]:
#prova punto 6 quanti aggettivi sinonimi del campione di 500 parole mi son rimasti?
len(synonym_pairs_test)

2423

In [39]:
#prova punto 7 riordino in ordine alfabtico il set
synonym_pairs_test = sorted(synonym_pairs_test)

In [42]:
#prova punto 3 prendiamo un campio ne di 500 parole


sample_synonym_pairs_test = random.sample(synonym_pairs_test, 500)

print(len(sample_synonym_pairs_test))

500


In [43]:
#prova punto 7 funzione per escludere parole troppo simili (fatigued-fatigue)

def too_similar(word1, word2):

    similarity = difflib.SequenceMatcher(
        None,
        word1,
        word2
    ).ratio()

    return similarity > 0.75

In [44]:
#prova punto 8 creo lista filtrata di aggettivi-sinonimi
clean_synonym_pairs = []

for w1, w2 in sample_synonym_pairs_test:

    if not too_similar(w1, w2):
        clean_synonym_pairs.append((w1, w2))

In [45]:
#prova punto 9 quanti ne rimangono?
len(clean_synonym_pairs)

421

In [46]:
#prova punto 10 guardiamo un campione
for pair in clean_synonym_pairs[:100]:
    print(pair)

('northward', 'northbound')
('revolved', 'rotated')
('ravaged', 'raped')
('dandy', 'nifty')
('afloat', 'aimless')
('petulant', 'testy')
('loath', 'reluctant')
('thumping', 'banging')
('master', 'chief')
('ultra', 'radical')
('disruptive', 'turbulent')
('meteorological', 'meteoric')
('suffocating', 'smothering')
('ghastly', 'macabre')
('dodgy', 'dicey')
('biweekly', 'fortnightly')
('pouring', 'gushing')
('sceptical', 'doubting')
('wed', 'wedded')
('superimposed', 'overlying')
('gilded', 'gilt')
('resultant', 'accompanying')
('jerky', 'jerking')
('precipitous', 'precipitate')
('dismayed', 'shocked')
('depicted', 'portrayed')
('multiplex', 'manifold')
('circular', 'round')
('sounding', 'looking')
('vivid', 'lifelike')
('prim', 'dainty')
('implemented', 'enforced')
('prevailing', 'rife')
('subversive', 'insurgent')
('demonic', 'satanic')
('desirous', 'wishful')
('unwavering', 'steadfast')
('firstborn', 'eldest')
('predominant', 'rife')
('fierce', 'ferocious')
('uptight', 'edgy')
('xviii', 

In [ ]:
remove_pairs = [

    ('northward', 'northbound'),
    # non veri sinonimi: northward = direzione verso nord, northbound = diretto verso nord

    ('ravaged', 'raped'),
    # eliminare: termine problematico e non sinonimo generale; "raped" ha significato specifico

    ('dandy', 'nifty'),
    # slang/informale; sinonimia debole

    ('afloat', 'aimless'),
    # significati diversi: afloat = galleggiante/solvibile, aimless = senza scopo

    ('thumping', 'banging'),
    # troppo dipendente dal contesto; spesso suoni/azioni diverse

    ('master', 'chief'),
    # polisemia elevata; master non equivale generalmente a chief

    ('ultra', 'radical'),
    # sinonimia solo in alcuni contesti politici/sociali

    ('meteorological', 'meteoric'),
    # falsi amici: meteorologico vs rapidissimo/brillante

    ('wed', 'wedded'),
    # stessa radice/variante morfologica, non vera coppia sinonimica

    ('superimposed', 'overlying'),
    # molto vicini ma descrivono relazioni spaziali diverse

    ('gilded', 'gilt'),
    # stessa famiglia morfologica, non sinonimi puliti

    ('resultant', 'accompanying'),
    # significati diversi

    ('jerky', 'jerking'),
    # stessa parola in forme diverse

    ('precipitous', 'precipitate'),
    # derivati simili ma non sinonimi affidabili

    ('multiplex', 'manifold'),
    # significato diverso: multiplex = multiplo/complesso, manifold = numeroso/diverso

    ('sounding', 'looking'),
    # non sinonimi

    ('prim', 'dainty'),
    # vicini solo in alcuni contesti

    ('subversive', 'insurgent'),
    # correlati ma non sinonimi

    ('demonic', 'satanic'),
    # molto vicini ma con forte componente culturale/religiosa; troppo specifici

    ('xviii', 'eighteen'),
    # numerale romano vs parola numerica

    ('resonant', 'resounding'),
    # troppo simili morfologicamente

    ('wee', 'bitty'),
    # wee = piccolo, bitty = frammentato; sinonimia debole

    ('xxx', 'thirty'),
    # numerale romano/forma non linguistica

    ('noted', 'famed'),
    # noted può significare osservato/notato, non solo famoso

    ('flaming', 'bally'),
    # bally è slang britannico raro

    ('downwind', 'lee'),
    # termini tecnici diversi

    ('blooming', 'flaming'),
    # non sinonimi; entrambi possono essere intensificatori ma senso diverso

    ('islamic', 'moslem'),
    # eliminare: termine controverso/meno moderno e non equivalente perfetto

    ('snobbish', 'snobby'),
    # stessa parola con variazione ortografica

    ('molded', 'wrought'),
    # significati diversi

    ('twilight', 'dusky'),
    # associati ma non sinonimi

    ('pedigree', 'thoroughbred'),
    # correlati ma non sinonimi

    ('baseless', 'unfounded'),
    # troppo vicini ma praticamente duplicati rispetto a groundless/unfounded

    ('husky', 'burly'),
    # husky ha molti significati (voce, cane, corporatura)

    ('encompassing', 'blanket'),
    # blanket solo metaforico in alcuni casi

    ('fleeting', 'fugitive'),
    # fugitive è soprattutto "latitante"; sinonimia debole

    ('adrift', 'afloat'),
    # adrift = senza direzione, afloat = galleggiante; non equivalenti

    ('poached', 'boiled'),
    # metodi di cottura diversi

    ('vain', 'swollen'),
    # significati diversi

]

In [47]:
#prova punto 12 guardiamo un campione
for pair in clean_synonym_pairs[100:200]:
    print(pair)

('counter', 'antagonistic')
('amok', 'berserk')
('kindred', 'akin')
('pleading', 'imploring')
('greyish', 'gray')
('pert', 'impertinent')
('swinging', 'tripping')
('conjugate', 'coupled')
('squalid', 'sordid')
('transmitted', 'inherited')
('extremist', 'radical')
('flagging', 'drooping')
('submarine', 'undersea')
('shocked', 'dismayed')
('baked', 'scorched')
('highland', 'upland')
('touching', 'affecting')
('waterlogged', 'marshy')
('motley', 'assorted')
('statute', 'codified')
('vengeful', 'vindictive')
('nonsensical', 'preposterous')
('numeric', 'numeral')
('obstinate', 'stubborn')
('gymnastic', 'acrobatic')
('inherent', 'integral')
('bridal', 'spousal')
('iii', 'three')
('pledged', 'sworn')
('archetypal', 'prototypical')
('integral', 'inherent')
('goddamned', 'blasted')
('nautical', 'maritime')
('detrimental', 'damaging')
('cognizant', 'aware')
('petite', 'tiny')
('therapeutic', 'healing')
('granted', 'given')
('harrowing', 'torturing')
('dizzy', 'giddy')
('renewing', 'restorative')

In [48]:
remove_pairs_2 = [

    ('counter', 'antagonistic'),
    # counter = contrario/opposto o bancone; antagonistic = ostile. Non sinonimi.

    ('swinging', 'tripping'),
    # significati diversi; tripping ha molti sensi non collegati.

    ('conjugate', 'coupled'),
    # conjugate è soprattutto grammaticale/scientifico; coupled = unito.

    ('transmitted', 'inherited'),
    # trasmesso vs ereditato: concetti diversi.

    ('extremist', 'radical'),
    # correlati ma non sinonimi perfetti; extremist è più forte.

    ('submarine', 'undersea'),
    # sottomarino vs relativo al mare profondo.

    ('baked', 'scorched'),
    # cottura vs bruciato; non equivalenti.

    ('highland', 'upland'),
    # molto vicini geograficamente ma non sinonimi perfetti.

    ('touching', 'affecting'),
    # touching = commovente o toccare; affecting = influente/commovente.

    ('waterlogged', 'marshy'),
    # impregnato d'acqua vs paludoso.

    ('statute', 'codified'),
    # legge/codice vs codificato.

    ('numeric', 'numeral'),
    # numerico vs cifra/simbolo numerico.

    ('gymnastic', 'acrobatic'),
    # attività diverse.

    ('inherent', 'integral'),
    # vicini ma non sinonimi: inherent = intrinseco, integral = parte essenziale.

    ('iii', 'three'),
    # numero romano.

    ('goddamned', 'blasted'),
    # contiene parolaccia/termine colloquiale.

    ('granted', 'given'),
    # troppo generici e polisemia elevata.

    ('harrowing', 'torturing'),
    # harrowing = straziante, torturing = torturare; non equivalenti.

    ('magic', 'wizard'),
    # magia vs mago.

    ('miscellaneous', 'motley'),
    # simili ma non sinonimi puliti.

    ('scurvy', 'abject'),
    # scurvy ha significato specifico/arcaico.

    ('departed', 'bygone'),
    # departed = partito/morto; bygone = passato.

    ('devouring', 'avid'),
    # azione vs caratteristica.

    ('flowered', 'floral'),
    # forma verbale vs aggettivo.

    ('deviate', 'aberrant'),
    # verbo vs aggettivo.

    ('through', 'done'),
    # non sinonimi.

    ('westward', 'westbound'),
    # direzione vs movimento verso ovest.

    ('bygone', 'departed'),
    # stesso problema invertito.

    ('treble', 'soprano'),
    # concetti musicali diversi.

    ('cutting', 'stinging'),
    # vicini solo in alcuni contesti.

    ('mass', 'aggregated'),
    # massa vs aggregato.

    ('rambling', 'sprawling'),
    # possono sovrapporsi ma non sinonimi.

    ('linear', 'additive'),
    # concetti matematici diversi.

    ('satanic', 'demonic'),
    # molto vicini ma troppo legati a concetti religiosi specifici.

    ('grey', 'grayish'),
    # variazione + derivazione, non sinonimo pulito.

    ('xix', 'nineteen'),
    # numero romano.

    ('any', 'whatever'),
    # non sinonimi perfetti.

    ('only', 'lonesome'),
    # completamente diversi.

    ('cheering', 'comforting'),
    # incoraggiante vs confortante.

    ('sovereign', 'autonomous'),
    # sovrano vs autonomo.

    ('wry', 'ironic'),
    # vicini ma wry ha sfumatura specifica.

    ('smashing', 'groovy'),
    # slang.

    ('voluptuous', 'luscious'),
    # vicini ma connotazioni diverse.

    ('plowed', 'ploughed'),
    # variante ortografica.

    ('stereotyped', 'stereotypical'),
    # stessa radice, non sinonimi.

    ('solemn', 'grave'),
    # grave ha molti significati; sinonimia solo parziale.

    ('developing', 'underdeveloped'),
    # opposti/parzialmente correlati.

    ('eight', 'viii'),
    # numero romano.

    ('aspirant', 'aspiring'),
    # stessa famiglia morfologica.

    ('agrarian', 'farming'),
    # correlati ma non sinonimi.

    ('given', 'granted'),
    # duplicato della coppia precedente.

    ('foldable', 'folding'),
    # stessa radice, significati diversi.
    
    ('greyish', 'gray'),
    # troppo vicini morfologicamente: greyish = leggermente grigio, gray = grigio.

    ('incorporate', 'integrated'),
    # stessa area semantica ma categorie/uso diversi:

]

In [49]:
#prova punto 12 guardiamo un campione
for pair in clean_synonym_pairs[200:300]:
    print(pair)

('epistemic', 'epistemological')
('round', 'circular')
('unrivalled', 'peerless')
('mercurial', 'quicksilver')
('shivering', 'trembling')
('hibernating', 'dormant')
('reprehensible', 'criminal')
('magyar', 'hungarian')
('precipitate', 'precipitous')
('tilted', 'leaning')
('aware', 'cognizant')
('unhurt', 'unharmed')
('coy', 'demure')
('sensed', 'perceived')
('sheared', 'shorn')
('barbed', 'biting')
('heroic', 'epic')
('tertiary', 'third')
('predominate', 'overriding')
('baffled', 'bewildered')
('ploughed', 'plowed')
('telling', 'telltale')
('wonky', 'awry')
('farming', 'agrarian')
('judicious', 'heady')
('rousing', 'stirring')
('presumptuous', 'assuming')
('loaded', 'laden')
('silly', 'goofy')
('elliptical', 'oval')
('labored', 'strained')
('vitriolic', 'caustic')
('tending', 'apt')
('dirty', 'soiled')
('situated', 'placed')
('sensible', 'reasonable')
('unspoken', 'mute')
('crippled', 'game')
('everlasting', 'eternal')
('desperate', 'despairing')
('powdered', 'pulverized')
('radical', 

In [50]:
remove_pairs_3 = [

    ('epistemic', 'epistemological'),
    # stessa radice ma significati diversi: epistemic = relativo alla conoscenza,
    # epistemological = relativo alla teoria della conoscenza.

    ('round', 'circular'),
    # spesso sinonimi, ma round è molto più ampio (forma, movimento, rotondo).

    ('mercurial', 'quicksilver'),
    # quicksilver è mercurio/metafora; non sinonimo generale.

    ('reprehensible', 'criminal'),
    # riprovevole vs criminale: non equivalenti.

    ('magyar', 'hungarian'),
    # stesso concetto ma Magyar è termine etnico/nazionale specifico.

    ('precipitate', 'precipitous'),
    # stessa radice ma significati diversi.

    ('sheared', 'shorn'),
    # stessa forma verbale/participio.

    ('barbed', 'biting'),
    # solo metaforicamente simili.

    ('heroic', 'epic'),
    # correlati ma non sinonimi.

    ('tertiary', 'third'),
    # tecnico vs numero ordinale; non sinonimia pulita.

    ('predominate', 'overriding'),
    # significati vicini ma non equivalenti.

    ('ploughed', 'plowed'),
    # variante ortografica.

    ('telling', 'telltale'),
    # telling = rivelatore; telltale = indicativo/traditore.
    # troppo vicini ma non sinonimi perfetti.

    ('wonky', 'awry'),
    # wonky = instabile/difettoso; awry = storto/sbagliato.

    ('farming', 'agrarian'),
    # attività vs relativo all'agricoltura.

    ('judicious', 'heady'),
    # significati diversi.

    ('presumptuous', 'assuming'),
    # assuming può essere "presuntuoso", ma ha altri sensi.

    ('loaded', 'laden'),
    # molto simili ma loaded è molto più polisemico.

    ('elliptical', 'oval'),
    # geometria vicina ma non sinonimi.

    ('crippled', 'game'),
    # game = zoppicante solo in senso raro/arcaico.

    ('desperate', 'despairing'),
    # stessa radice ma significato leggermente diverso.

    ('radical', 'extremist'),
    # correlati ma extremist è più forte.

    ('westbound', 'westward'),
    # direzione vs movimento verso ovest.

    ('select', 'prize'),
    # select = scelto, prize = premio/prezioso.

    ('antagonistic', 'counter'),
    # non sinonimi.

    ('set', 'primed'),
    # significati diversi.

    ('sewn', 'sewed'),
    # stessa parola in forme grammaticali diverse.

    ('eighteen', 'xviii'),
    # numero romano.

    ('cruel', 'barbarous'),
    # molto vicini ma barbarous ha anche senso "incivile".

    ('taking', 'fetching'),
    # taking = prendere, fetching = attraente/andare a prendere.

    ('transferable', 'moveable'),
    # trasferibile vs spostabile.

    ('bastard', 'phoney'),
    # bastard è offensivo e ha altri significati.

    ('dissected', 'cleft'),
    # azione chirurgica vs diviso.

    ('recognized', 'accepted'),
    # riconosciuto vs accettato.

    ('scorched', 'parched'),
    # bruciato vs secco/disidratato.

    ('jammed', 'packed'),
    # bloccato vs pieno.

    ('curving', 'curved'),
    # stessa radice/forma.

    ('trespassing', 'encroaching'),
    # invasione fisica vs sconfinamento; vicini ma non perfetti.

    ('humbled', 'humiliated'),
    # umiliato vs reso umile; sfumatura diversa.

    ('aged', 'elderly'),
    # aged = invecchiato/vecchio; elderly = anziano.

    ('stung', 'pissed'),
    # pissed è slang (anche volgare).

    ('stillborn', 'abortive'),
    # nati morti vs fallito/incompiuto.

    ('halting', 'halt'),
    # stessa radice grammaticale.

    ('garish', 'flash'),
    # flash = vistoso solo in slang.

    ('sorry', 'regretful'),
    # sorry è molto più ampio.

    ('punk', 'cheesy'),
    # slang e significati diversi.

    ('twisting', 'winding'),
    # azione/processo vs forma.

    ('additive', 'linear'),
    # concetti matematici diversi.

    ('foster', 'surrogate'),
    # foster = affidatario, surrogate = sostitutivo.

    ('dismissed', 'fired'),
    # fired è solo un senso specifico di dismissed.

    ('triumphant', 'rejoicing'),
    # vittorioso vs gioioso.

    ('swampy', 'waterlogged'),
    # paludoso vs impregnato d'acqua.

]

In [51]:
#prova punto 12 guardiamo un campione
for pair in clean_synonym_pairs[300:]:
    print(pair)

('erstwhile', 'sometime')
('eastbound', 'eastward')
('introspective', 'introverted')
('epistemological', 'epistemic')
('shaggy', 'bushy')
('urgent', 'pressing')
('attached', 'connected')
('leery', 'suspicious')
('meddling', 'interfering')
('overseas', 'abroad')
('permanent', 'lasting')
('unclean', 'soiled')
('repetitive', 'insistent')
('bubbly', 'foaming')
('kinky', 'perverted')
('snide', 'sneering')
('bust', 'broke')
('branched', 'forked')
('northbound', 'northward')
('resettled', 'relocated')
('unique', 'unparalleled')
('eldest', 'firstborn')
('impregnable', 'unassailable')
('forked', 'branched')
('implanted', 'ingrained')
('conversational', 'colloquial')
('zippy', 'spanking')
('convalescent', 'recovering')
('loco', 'daft')
('whitish', 'milky')
('outlaw', 'illegitimate')
('static', 'motionless')
('mammoth', 'gigantic')
('improvised', 'makeshift')
('teeny', 'bitty')
('hazardous', 'risky')
('ordained', 'decreed')
('unquestioned', 'unchallenged')
('sleepless', 'insomniac')
('transcenden

In [52]:
remove_pairs_4 = [

    ('erstwhile', 'sometime'),
    # sinonimia debole: erstwhile = ex/precedente, sometime = in qualche momento.

    ('eastbound', 'eastward'),
    # direzione vs movimento verso est.

    ('introspective', 'introverted'),
    # riflessivo/interiore vs persona socialmente ritirata.

    ('epistemological', 'epistemic'),
    # stessa radice ma concetti diversi.

    ('attached', 'connected'),
    # attaccato fisicamente/emotivamente vs connesso.

    ('permanent', 'lasting'),
    # lasting è più generico.

    ('repetitive', 'insistent'),
    # ripetitivo vs insistente.

    ('kinky', 'perverted'),
    # kinky è slang/ambiguo; perverted ha connotazioni diverse.

    ('bust', 'broke'),
    # slang: bust = fallire/rompere, broke = senza soldi.

    ('unique', 'unparalleled'),
    # unico vs senza paragoni, vicini ma non sinonimi perfetti.

    ('zippy', 'spanking'),
    # slang/uso particolare.

    ('whitish', 'milky'),
    # colore simile ma non sinonimi.

    ('outlaw', 'illegitimate'),
    # fuorilegge vs illegittimo.

    ('static', 'motionless'),
    # statico è più ampio.

    ('discarded', 'throwaway'),
    # scartato vs usa-e-getta.

    ('demeaning', 'humbling'),
    # umiliante negativo vs rendere umile.

    ('irritating', 'pesky'),
    # pesky è colloquiale e più specifico.

    ('macabre', 'gruesome'),
    # vicini ma non equivalenti.

    ('subdued', 'muted'),
    # attenuato/sottomesso vs silenziato.

    ('interconnected', 'interrelated'),
    # molto simili ma sfumatura diversa.

    ('outlandish', 'gonzo'),
    # gonzo è stile giornalistico/slang.

    ('manifest', 'apparent'),
    # evidente vs manifesto.

    ('projecting', 'protruding'),
    # proiettare vs sporgere.

    ('swish', 'classy'),
    # swish è slang britannico.

    ('unsatisfying', 'disappointing'),
    # insoddisfacente vs deludente.

    ('obscure', 'vague'),
    # oscuro/ignoto vs poco chiaro.

    ('underdeveloped', 'developing'),
    # non sviluppato vs in sviluppo.

    ('conciliatory', 'compromising'),
    # conciliante vs disposto al compromesso.

    ('abiding', 'enduring'),
    # abiding = duraturo/rispettato; enduring = resistente.

    ('agape', 'gaping'),
    # agape = a bocca aperta; gaping = spalancato.

    ('gritty', 'granular'),
    # consistenza/texture diverse.

    ('pretended', 'assumed'),
    # fingere vs assumere.

    ('contraband', 'smuggled'),
    # merce illegale vs contrabbandato.

    ('primal', 'cardinal'),
    # fondamentale in sensi diversi.

    ('looking', 'sounding'),
    # percezione visiva vs uditiva.

    ('vii', 'seven'),
    # numero romano.

    ('emaciated', 'bony'),
    # magro/denutrito vs ossuto.

    ('unmanageable', 'unwieldy'),
    # difficile da gestire vs ingombrante.

    ('synergistic', 'interactive'),
    # collaborazione positiva vs interazione.

    ('materialistic', 'mercenary'),
    # materialista vs interessato al denaro.

    ('consonant', 'harmonized'),
    # consonante linguistica vs armonizzato.

    ('degraded', 'libertine'),
    # degradato vs libertino.

    ('assuming', 'presumptuous'),
    # assuming ha più significati.

    ('domed', 'vaulted'),
    # forme architettoniche vicine ma diverse.

    ('teen', 'teenaged'),
    # stessa informazione in forme diverse.

    ('humbling', 'humiliating'),
    # sfumatura diversa.

    ('eccentric', 'outlandish'),
    # eccentrico vs assurdo.

    ('pictured', 'envisioned'),
    # immaginato vs rappresentato.

    ('accepted', 'recognized'),
    # accettato vs riconosciuto.

    ('bonny', 'bonnie'),
    # variante ortografica.

    ('needy', 'destitute'),
    # bisognoso vs poverissimo.

    ('elongated', 'lengthened'),
    # stessa azione/derivazione.

    ('teenaged', 'teen'),
    # stessa radice.

    ('winding', 'twisty'),
    # molto vicini ma non sinonimi puliti.

    ('introverted', 'introspective'),
    # duplicato già eliminato.

    ('wizard', 'magical'),
    # persona vs qualità.

    ('hated', 'scorned'),
    # odio vs disprezzo.

    ('main', 'master'),
    # principale vs maestro.

    ('returning', 'reverting'),
    # tornare vs ritornare allo stato precedente.

    ('indentured', 'apprenticed'),
    # lavoratore vincolato vs apprendista.

    ('shaggy', 'bushy'),
    # entrambi indicano "folto", ma shaggy = trasandato/lungo,
    # bushy = folto; non intercambiabili.

    ('urgent', 'pressing'),
    # pressing è spesso "incalzante", urgent è "urgente".

    ('leery', 'suspicious'),
    # leery = diffidente/cauto, suspicious = sospettoso.
    # Stessa area ma diversa prospettiva.

    ('meddling', 'interfering'),
    # quasi sinonimi ma meddling implica intromissione indesiderata.

    ('overseas', 'abroad'),
    # entrambi "all'estero", ma overseas ha anche senso geografico specifico.

    ('permanent', 'lasting'),
    # già eliminata: lasting è troppo generico.

    ('mammoth', 'gigantic'),
    # mammoth è soprattutto metaforico "enorme"; gigantic è dimensione pura.

    ('improvised', 'makeshift'),
    # improvised = creato sul momento,
    # makeshift = temporaneo/provvisorio.

    ('hazardous', 'risky'),
    # rischio vs pericolosità; molto vicini ma non equivalenti.

    ('objectionable', 'obnoxious'),
    # objectionable = criticabile,
    # obnoxious = fastidioso/offensivo.

    ('furious', 'fierce'),
    # furious = arrabbiato,
    # fierce = aggressivo/intenso.

    ('thriving', 'prospering'),
    # quasi sinonimi ma thriving implica crescita/vitalità.

    ('immaculate', 'spotless'),
    # spotless è pulizia fisica,
    # immaculate è anche perfezione morale/assoluta.

    ('whispering', 'murmuring'),
    # whispering = parlare piano,
    # murmuring = borbottare/mormorare.

    ('wounded', 'hurt'),
    # wounded è ferito fisicamente,
    # hurt è molto più ampio.

    ('cured', 'healed'),
    # cured può essere eliminazione di malattia,
    # healed include anche guarigione emotiva/fisica.

    ('sterile', 'infertile'),
    # sterile è più generale (anche ambienti/strumenti),
    # infertile solo capacità riproduttiva.

    ('fortnightly', 'biweekly'),
    # ambiguità: biweekly può essere due volte a settimana o ogni due settimane.

    ('proficient', 'practiced'),
    # proficient = competente,
    # practiced = esercitato/esperto.

    ('unbounded', 'boundless'),
    # quasi sinonimi ma molto letterari.

    ('cornered', 'trapped'),
    # cornered implica essere messi alle strette,
    # trapped essere intrappolati.

    ('nimble', 'spry'),
    # nimble = agile,
    # spry = vivace/energico soprattutto per persone anziane.

    ('predictive', 'prognostic'),
    # predictive è generale,
    # prognostic è soprattutto medico.

    ('rattled', 'flustered'),
    # rattled = agitato/scosso,
    # flustered = confuso.

]


In [58]:
filtered_synonym_pairs = [
    pair for pair in clean_synonym_pairs
    if pair not in remove_pairs and pair not in remove_pairs_2 and pair not in remove_pairs_3 and pair not in remove_pairs_4
]

print(len(clean_synonym_pairs))
print(len(filtered_synonym_pairs))

421
231


In [59]:
#elimino i duplicati: jelous-envious envious-jelous

unique_pairs = []
seen = set()

for w1, w2 in filtered_synonym_pairs:

    key = tuple(sorted((w1, w2)))

    if key not in seen:
        seen.add(key)
        unique_pairs.append((w1, w2))

filtered_synonym_pairs = unique_pairs

print(len(filtered_synonym_pairs))

219


In [60]:
#controllo finale di sinonimi
for pair in filtered_synonym_pairs:
    print(pair)

('northward', 'northbound')
('revolved', 'rotated')
('ravaged', 'raped')
('dandy', 'nifty')
('afloat', 'aimless')
('petulant', 'testy')
('loath', 'reluctant')
('thumping', 'banging')
('master', 'chief')
('ultra', 'radical')
('disruptive', 'turbulent')
('meteorological', 'meteoric')
('suffocating', 'smothering')
('ghastly', 'macabre')
('dodgy', 'dicey')
('biweekly', 'fortnightly')
('pouring', 'gushing')
('sceptical', 'doubting')
('wed', 'wedded')
('superimposed', 'overlying')
('gilded', 'gilt')
('resultant', 'accompanying')
('jerky', 'jerking')
('precipitous', 'precipitate')
('dismayed', 'shocked')
('depicted', 'portrayed')
('multiplex', 'manifold')
('circular', 'round')
('sounding', 'looking')
('vivid', 'lifelike')
('prim', 'dainty')
('implemented', 'enforced')
('prevailing', 'rife')
('subversive', 'insurgent')
('demonic', 'satanic')
('desirous', 'wishful')
('unwavering', 'steadfast')
('firstborn', 'eldest')
('predominant', 'rife')
('fierce', 'ferocious')
('uptight', 'edgy')
('xviii', 

In [62]:
remove_pairs_final = [

('ravaged', 'raped'),        # volgare; significato non equivalente (raped è specifico)
('afloat', 'aimless'),       # non sinonimi: "a galla" vs "senza direzione"
('loath', 'reluctant'),      # quasi sinonimi ma sfumatura diversa: riluttante ≠ provare disgusto
('thumping', 'banging'),     # molto colloquiale, significati diversi
('master', 'chief'),         # relazione gerarchica, non sinonimi puri
('ultra', 'radical'),        # ultra può significare estremo ma non sempre radical
('meteorological', 'meteoric'), # falso amico, significati diversi
('wed', 'wedded'),            # stessa radice, forma grammaticale diversa
('gilded', 'gilt'),           # stesso campo ma non sinonimi perfetti
('resultant', 'accompanying'), # relazione, non sinonimi
('jerky', 'jerking'),         # derivazione grammaticale
('precipitous', 'precipitate'), # parole correlate ma non sinonimi
('dismayed', 'shocked'),      # shock più ampio
('multiplex', 'manifold'),    # significati diversi
('sounding', 'looking'),      # non sinonimi
('prim', 'dainty'),           # sfumature diverse
('prevailing', 'rife'),       # non sempre equivalenti
('predominant', 'rife'),      # frequenza vs predominanza
('xviii', 'eighteen'),        # numero romano
('xxx', 'thirty'),            # numero romano
('flaming', 'bally'),         # bally raro/arcaico
('downwind', 'lee'),          # relazione spaziale
('blooming', 'flaming'),      # non sinonimi
('islamic', 'moslem'),        # termine religioso/etnico, non utile semanticamente
('snobbish', 'snobby'),       # stessa parola, variante
('molded', 'wrought'),        # non equivalenti
('pedigree', 'thoroughbred'), # pedigree ≠ thoroughbred
('encompassing', 'blanket'),  # blanket come metafora, troppo distante
('adrift', 'afloat'),         # parziale, non sinonimi puri
('poached', 'boiled'),        # tecniche di cottura diverse
('vain', 'swollen'),          # solo un significato secondario
('pert', 'impertinent'),      # impertinent ha connotazione negativa diversa
('bridal', 'spousal'),        # bridal riguarda matrimonio, spousal riguarda coniuge
('renewing', 'restorative'),  # relazione, non sinonimi
('petite', 'tiny'),           # piccolo vs minuta (vicini ma diversi)
('unassuming', 'retiring'),   # sfumature diverse
('disgraced', 'dishonored'),  # molto simili ma connotazioni diverse
('unnamed', 'nameless'),     # quasi identici ma poco informativi
('dented', 'crumpled'),      # tipi diversi di deformazione
('appalled', 'shocked'),     # shock troppo generico
('fearsome', 'terrible'),    # terrible troppo generale
('unhurt', 'unharmed'),      # sinonimi validi ma molto banali
('sensed', 'perceived'),     # verbo vs percezione più ampia
('tending', 'apt'),          # non sinonimi
('situated', 'placed'),      # relazione di posizione, non sinonimi
('established', 'constituted'), # non equivalenti
('piercing', 'incisive'),    # metafora diversa
('moved', 'stirred'),        # troppo polisemi
('loony', 'kooky'),          # colloquiale/slang
('accompanying', 'concomitant'), # troppo tecnico
('helpless', 'incapacitated'),   # causa vs stato
('insipid', 'bland'),        # validi ma molto vicini a "taste"; valuterei rimozione
('learned', 'erudite'),      # learned ambiguo (imparato/dotto)
('unimaginable', 'inconceivable'), # validi ma quasi astratti
('bubbly', 'foaming'),       # persona allegra vs schiumoso
('loco', 'daft'),             # slang
('sleepless', 'insomniac'),   # stato vs persona
('transcendental', 'otherworldly'), # filosofia vs metafora
('stringy', 'wiry'),          # vicini ma non equivalenti
('infuriated', 'angered'),    # quasi sinonimi ma stessa famiglia emotiva
('silky', 'silken'),          # derivazione
('halfway', 'center'),       # non sinonimi
('shorn', 'sheared'),        # forma grammaticale
]

In [63]:
filtered_synonym_pairs_1 = [
    pair for pair in filtered_synonym_pairs
    if pair not in remove_pairs_final
]

print(len(clean_synonym_pairs))
print(len(filtered_synonym_pairs))
print(len(filtered_synonym_pairs_1))

421
219
156


In [64]:
#controllo finale di sinonimi
for pair in filtered_synonym_pairs_1:
    print(pair)

('northward', 'northbound')
('revolved', 'rotated')
('dandy', 'nifty')
('petulant', 'testy')
('disruptive', 'turbulent')
('suffocating', 'smothering')
('ghastly', 'macabre')
('dodgy', 'dicey')
('biweekly', 'fortnightly')
('pouring', 'gushing')
('sceptical', 'doubting')
('superimposed', 'overlying')
('depicted', 'portrayed')
('circular', 'round')
('vivid', 'lifelike')
('implemented', 'enforced')
('subversive', 'insurgent')
('demonic', 'satanic')
('desirous', 'wishful')
('unwavering', 'steadfast')
('firstborn', 'eldest')
('fierce', 'ferocious')
('uptight', 'edgy')
('headlong', 'hasty')
('captivated', 'charmed')
('incisive', 'penetrating')
('resonant', 'resounding')
('wee', 'bitty')
('credible', 'believable')
('raiding', 'marauding')
('reinforced', 'strengthened')
('outmoded', 'passe')
('fascinating', 'gripping')
('annoying', 'galling')
('noted', 'famed')
('saintly', 'angelic')
('disguised', 'masked')
('ruinous', 'catastrophic')
('beefy', 'burly')
('groundless', 'unfounded')
('passing', '

In [65]:
remove_pairs_final_2 = [

('northward', 'northbound'), 
# direzione geografica: "verso nord" vs "diretto verso nord"; quasi sinonimi ma non equivalenti

('dandy', 'nifty'),
# dandy = elegante/raffinato (persona), nifty = bello/pratico; senso diverso

('petulant', 'testy'),
# entrambi irritabili ma petulant = infantile/capriccioso, testy = facilmente irritabile

('disruptive', 'turbulent'),
# disruptive = che causa problemi; turbulent = caotico/instabile

('suffocating', 'smothering'),
# molto vicini ma spesso suffocating = atmosfera/oppressione, smothering = soffocare fisicamente

('raiding', 'marauding'),
# raid = incursione specifica, marauding = vagare saccheggiando

('saintly', 'angelic'),
# area semantica simile ma angelic è spesso estetico/metaforico

('beefy', 'burly'),
# beefy può riferirsi a muscoloso o massiccio; burly più fisico

('husky', 'burly'),
# husky può essere voce/cane/grande; troppo polisemo

('passing', 'transitory'),
# passing ha molti sensi (morire, superare, passare)

('disloyal', 'unpatriotic'),
# disloyal = infedele; unpatriotic = non patriottico

('spouting', 'squirting'),
# spouting = emettere getto; squirting più specifico

('diminutive', 'tiny'),
# quasi sinonimi ma diminutive ha anche senso grammaticale/di forma

('fleeting', 'fugitive'),
# fugitive = in fuga, non semplicemente breve

('blinding', 'glaring'),
# blinding = che impedisce vista; glaring = evidente/intenso

('pleading', 'imploring'),
# molto vicini ma pleading spesso implica richiesta/giudizio

('motley', 'assorted'),
# motley = misto ma con connotazione disordinata

('pledged', 'sworn'),
# pledged = promesso/impegnato; sworn = giurato

('integral', 'inherent'),
# integral = parte necessaria; inherent = proprietà intrinseca

('cognizant', 'aware'),
# validi ma cognizant è molto formale; lo terrei borderline

('exonerated', 'vindicated'),
# simili ma vindicated implica dimostrazione di correttezza

('uncouth', 'vulgar'),
# uncouth = rozzo; vulgar = volgare/comune

('expressed', 'uttered'),
# expressed è molto più ampio

('grisly', 'ghastly'),
# grisly = macabro/sanguinoso, ghastly = orribile/spaventoso

('shivering', 'trembling'),
# tremble può essere per paura/emozione, shiver spesso freddo

('hibernating', 'dormant'),
# hibernating biologico specifico, dormant generale

('coy', 'demure'),
# molto vicini ma coy spesso implica comportamento studiato

('rousing', 'stirring'),
# entrambi evocativi ma non sempre sinonimi

('vitriolic', 'caustic'),
# caustic può essere chimico; vitriolic è aggressivo nel linguaggio

('unspoken', 'mute'),
# unspoken = non detto, mute = incapace di parlare

('powdered', 'pulverized'),
# powdered = trasformato in polvere; pulverized anche distrutto

('searching', 'probing'),
# searching molto generico

('ecstatic', 'rapt'),
# rapt può essere concentrato/assorbito, non solo felice

('bland', 'vapid'),
# molto vicini ma bland può essere gusto, vapid più insipido/intellettualmente vuoto

('lubricated', 'greased'),
# validi ma troppo dominio-specifici

('sagging', 'droopy'),
# sagging = cedimento fisico, droopy anche aspetto

('conversational', 'colloquial'),
# conversational = dialogico, colloquial = informale

('convalescent', 'recovering'),
# convalescent è una persona in recupero, recovering più generale

('teeny', 'bitty'),
# entrambi informali

('ordained', 'decreed'),
# ordained ha anche senso religioso

('unquestioned', 'unchallenged'),
# vicini ma non identici

('wounding', 'stabbing'),
# stabbing è specifico

('sunny', 'cheery'),
# sunny può essere meteo

('mundane', 'everyday'),
# mundane = ordinario/banale, everyday = quotidiano

('scheming', 'calculating')
# scheming ha connotazione negativa, calculating può essere neutro
]

In [66]:
filtered_synonym_pairs_final = [
    pair for pair in filtered_synonym_pairs_1
    if pair not in remove_pairs_final_2
]

print(len(clean_synonym_pairs))
print(len(filtered_synonym_pairs))
print(len(filtered_synonym_pairs_1))
print(len(filtered_synonym_pairs_final))

421
219
156
111


In [67]:
#controllo finale di sinonimi
for pair in filtered_synonym_pairs_final:
    print(pair)

('revolved', 'rotated')
('ghastly', 'macabre')
('dodgy', 'dicey')
('biweekly', 'fortnightly')
('pouring', 'gushing')
('sceptical', 'doubting')
('superimposed', 'overlying')
('depicted', 'portrayed')
('circular', 'round')
('vivid', 'lifelike')
('implemented', 'enforced')
('subversive', 'insurgent')
('demonic', 'satanic')
('desirous', 'wishful')
('unwavering', 'steadfast')
('firstborn', 'eldest')
('fierce', 'ferocious')
('uptight', 'edgy')
('headlong', 'hasty')
('captivated', 'charmed')
('incisive', 'penetrating')
('resonant', 'resounding')
('wee', 'bitty')
('credible', 'believable')
('reinforced', 'strengthened')
('outmoded', 'passe')
('fascinating', 'gripping')
('annoying', 'galling')
('noted', 'famed')
('disguised', 'masked')
('ruinous', 'catastrophic')
('groundless', 'unfounded')
('clothed', 'clad')
('weird', 'uncanny')
('healing', 'therapeutic')
('aghast', 'appalled')
('neglected', 'ignored')
('loathsome', 'nauseating')
('sedate', 'staid')
('angered', 'enraged')
('twilight', 'dusky'

In [69]:
dataset_sinonimi= [('revolved', 'rotated'),
('ghastly', 'macabre'),
('dodgy', 'dicey'),
('biweekly', 'fortnightly'),
('pouring', 'gushing'),
('sceptical', 'doubting'),
('superimposed', 'overlying'),
('depicted', 'portrayed'),
('circular', 'round'),
('vivid', 'lifelike'),
('implemented', 'enforced'),
('subversive', 'insurgent'),
('demonic', 'satanic'),
('desirous', 'wishful'),
('unwavering', 'steadfast'),
('firstborn', 'eldest'),
('fierce', 'ferocious'),
('uptight', 'edgy'),
('headlong', 'hasty'),
('captivated', 'charmed'),
('incisive', 'penetrating'),
('resonant', 'resounding'),
('wee', 'bitty'),
('credible', 'believable'),
('reinforced', 'strengthened'),
('outmoded', 'passe'),
('fascinating', 'gripping'),
('annoying', 'galling'),
('noted', 'famed'),
('disguised', 'masked'),
('ruinous', 'catastrophic'),
('groundless', 'unfounded'),
('clothed', 'clad'),
('weird', 'uncanny'),
('healing', 'therapeutic'),
('aghast', 'appalled'),
('neglected', 'ignored'),
('loathsome', 'nauseating'),
('sedate', 'staid'),
('angered', 'enraged'),
('twilight', 'dusky'),
('frugal', 'sparing'),
('underweight', 'scrawny'),
('thrilling', 'electrifying'),
('illustrious', 'famous'),
('unforeseen', 'unanticipated'),
('amalgamated', 'fused'),
('toiling', 'labouring'),
('baseless', 'unfounded'),
('bracing', 'refreshing'),
('intrepid', 'dauntless'),
('ineffective', 'ineffectual'),
('skilful', 'expert'),
('amok', 'berserk'),
('kindred', 'akin'),
('squalid', 'sordid'),
('flagging', 'drooping'),
('vengeful', 'vindictive'),
('nonsensical', 'preposterous'),
('obstinate', 'stubborn'),
('archetypal', 'prototypical'),
('nautical', 'maritime'),
('detrimental', 'damaging'),
('dizzy', 'giddy'),
('unavoidable', 'inescapable'),
('desolate', 'barren'),
('dreadful', 'horrendous'),
('marooned', 'stranded'),
('disgruntled', 'dissatisfied'),
('puzzling', 'enigmatic'),
('infertile', 'sterile'),
('faithless', 'traitorous'),
('deserved', 'merited'),
('exhausting', 'draining'),
('goofy', 'silly'),
('flustered', 'perturbed'),
('burdensome', 'taxing'),
('unrivalled', 'peerless'),
('tilted', 'leaning'),
('baffled', 'bewildered'),
('labored', 'strained'),
('dirty', 'soiled'),
('sensible', 'reasonable'),
('everlasting', 'eternal'),
('flourishing', 'prospering'),
('colossal', 'stupendous'),
('cloaked', 'masked'),
('drunken', 'boozy'),
('discouraged', 'disheartened'),
('evoked', 'elicited'),
('shouted', 'yelled'),
('striking', 'spectacular'),
('extended', 'prolonged'),
('disgraceful', 'scandalous'),
('coveted', 'desired'),
('razed', 'demolished'),
('unclean', 'soiled'),
('snide', 'sneering'),
('branched', 'forked'),
('resettled', 'relocated'),
('impregnable', 'unassailable'),
('implanted', 'ingrained'),
('rejoicing', 'jubilant'),
('dreary', 'drab'),
('enthralling', 'captivating'),
('tart', 'tangy'),
('agitating', 'provoking'),
('concluded', 'ended'),
('enamored', 'infatuated'),
('hardworking', 'industrious'),
('alluring', 'beguiling')]

In [ ]:
#creo file csv di dataset sinonimi
df = pd.DataFrame(
    dataset_sinonimi,
    columns=["word", "synonym"]
)



In [78]:
#creo colonna dove specifico la relazione
df["relation"] = "synonym"

In [79]:
df.to_csv(
    "synonym_pairs_final.csv",
    index=False
)

df.to_excel("synonym_pairs_final.xlsx", index=False)